In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


plt.style.use("seaborn-v0_8")

In [67]:
apartments = pd.read_csv("../data/sales.csv")
apartments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 822 entries, 0 to 821
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price_numeric  822 non-null    float64
 1   municipality   822 non-null    object 
 2   condition      822 non-null    object 
 3   rooms          822 non-null    float64
 4   square_m2      822 non-null    float64
 5   equipment      822 non-null    object 
 6   level          822 non-null    int64  
 7   heating        822 non-null    object 
 8   price_per_m2   822 non-null    float64
dtypes: float64(4), int64(1), object(4)
memory usage: 57.9+ KB


In [68]:
from sklearn.model_selection import train_test_split, cross_val_score

X = apartments.drop(labels=["price_numeric", "price_per_m2", "square_m2"], axis="columns")
y = apartments["price_per_m2"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X['municipality'])

In [69]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline

non_numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', "passthrough", ['level', 'rooms'])
], remainder="drop")

numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', StandardScaler(), ['level', 'rooms'])
], remainder="drop")

In [70]:
from sklearn.linear_model import LinearRegression

lin_reg_pipe_line = Pipeline(steps=[
  ('col_transform', numeric_CT),
  ('lin_reg', LinearRegression())
])

cross_val_score(lin_reg_pipe_line, X_train, y_train, cv=5).mean()

np.float64(0.24046931063755778)

In [71]:
from sklearn.ensemble import RandomForestRegressor
rand_forest_ensemble = Pipeline(steps=[
  ('col_transform', non_numeric_CT),
  ('rand_forest', RandomForestRegressor(random_state=42))
])

cross_val_score(rand_forest_ensemble, X_train, y_train, cv=5).mean()

np.float64(0.2006761738190911)

In [72]:
from sklearn.svm import SVR
svm_pipeline = Pipeline([
    ("preprocess", numeric_CT),   # includes StandardScaler for numeric
    ("svm", SVR(kernel="linear"))
])

cross_val_score(svm_pipeline, X_train, y_train, cv=5).mean()

np.float64(0.004559151766366098)

In [73]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps=[
    ("preprocess", numeric_CT),
    ("ridge", Ridge(alpha=1.0))
])
cross_val_score(ridge_pipeline, X_train, y_train, cv=5).mean()

np.float64(0.2409843881038618)